# 03 · Product Demand Forecasting

**Business question:** *How many units of each best-selling product will be sold per day?*

Better demand forecasts help decide how much stock to hold and when to reorder.

**Approach**

1. Load daily quantity per product from `dbo.vw_ProductDailyDemand`.
2. Keep the **20 best-selling products** and fill days without sales with zero demand.
3. Build features: calendar fields, **lags** (what sold 1, 7, 14, 28 days ago) and **rolling averages**.
4. Split **by time**: train on the past, test on the **last 30 days**.
5. Compare a simple baseline against Random Forest and XGBoost.

> The forecast is **one day ahead**: yesterday's real sales are available when predicting today. Forecasting a whole month ahead would need a recursive or direct multi-step approach.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Find the project root (the folder that contains "src"), whatever the working directory is
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "db.py").exists()), None)
if root is None:
    raise FileNotFoundError("Open this notebook from inside the retail-intelligence folder (it needs the src/ folder next to it).")
sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src.db import read_sql       # runs a SELECT on SQL Server and returns a DataFrame

plt.rcParams["figure.dpi"] = 100

import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

RANDOM_STATE = 42
TOP_N_PRODUCTS = 20
TEST_DAYS = 30

## 2. Load and inspect the data

In [ ]:
df_demand = read_sql("SELECT * FROM dbo.vw_ProductDailyDemand")
df_demand["SalesDate"] = pd.to_datetime(df_demand["SalesDate"])

print(f"Rows            : {len(df_demand):,}")
print(f"Products        : {df_demand['StockCode'].nunique():,}")
print(f"Date range      : {df_demand['SalesDate'].min().date()}  ->  {df_demand['SalesDate'].max().date()}")
print(f"Missing values  : {int(df_demand.isnull().sum().sum())}")
df_demand.head()

## 3. Select the products and complete the calendar

Forecasting all ~4,900 products would be dominated by slow sellers with almost no data, so the **20 products with the highest total quantity** are modelled.

The SQL view has **no row for a day when a product did not sell**. Those days are real zero-demand days, so the calendar is rebuilt for each product (from its first to its last sale) and the gaps are filled with 0.

In [ ]:
top_products = (
    df_demand.groupby("StockCode")["DailyQuantity"].sum()
    .nlargest(TOP_N_PRODUCTS).index
)

df_top = df_demand[df_demand["StockCode"].isin(top_products)]


def add_missing_days(df: pd.DataFrame) -> pd.DataFrame:
    """For every product, insert the days without sales with zero demand."""
    frames = []

    for product, product_df in df.groupby("StockCode", sort=False):
        product_df = product_df.set_index("SalesDate")
        all_days = pd.date_range(product_df.index.min(), product_df.index.max(), freq="D")

        product_df = product_df.reindex(all_days)
        product_df["StockCode"] = product
        product_df[["DailyQuantity", "DailyRevenue", "DailyOrders"]] = (
            product_df[["DailyQuantity", "DailyRevenue", "DailyOrders"]].fillna(0)
        )
        product_df.index.name = "SalesDate"
        frames.append(product_df.reset_index())

    return pd.concat(frames, ignore_index=True)


df_forecast = add_missing_days(df_top)

zero_days = (df_forecast["DailyQuantity"] == 0).mean()
print(f"Rows after filling the calendar: {len(df_forecast):,}  ({zero_days:.0%} of them are zero-demand days)")

## 4. Feature engineering

| Feature | Meaning |
|---|---|
| `DayOfWeek`, `DayOfMonth`, `Month`, `Year`, `IsWeekend` | Calendar effects |
| `Lag_1`, `Lag_7`, `Lag_14`, `Lag_28` | Demand 1, 7, 14 and 28 days earlier |
| `RollingMean_7`, `RollingMean_28` | Average demand over the previous 7 / 28 days |

All lag and rolling features are computed **per product** and **shifted by one day**, so a row never uses its own day's sales (no leakage).

In [ ]:
# Calendar features
df_forecast["DayOfWeek"] = df_forecast["SalesDate"].dt.dayofweek      # Monday = 0
df_forecast["DayOfMonth"] = df_forecast["SalesDate"].dt.day
df_forecast["Month"] = df_forecast["SalesDate"].dt.month
df_forecast["Year"] = df_forecast["SalesDate"].dt.year
df_forecast["IsWeekend"] = (df_forecast["DayOfWeek"] >= 5).astype(int)

# Lag features (per product)
by_product = df_forecast.groupby("StockCode")["DailyQuantity"]
for lag in (1, 7, 14, 28):
    df_forecast[f"Lag_{lag}"] = by_product.shift(lag)

# Rolling averages of the PREVIOUS days (shift(1) keeps the current day out)
for window in (7, 28):
    df_forecast[f"RollingMean_{window}"] = by_product.transform(
        lambda s: s.shift(1).rolling(window).mean()
    )

# The first 28 days of each product have no full history -> drop them
lag_columns = ["Lag_1", "Lag_7", "Lag_14", "Lag_28", "RollingMean_7", "RollingMean_28"]
df_forecast = df_forecast.dropna(subset=lag_columns).reset_index(drop=True)

print(f"Modelling rows : {len(df_forecast):,}")
print(f"Products kept  : {df_forecast['StockCode'].nunique()}  (a product needs more than 28 days of history)")

## 5. Time-based train / test split

The **last 30 days** are held out as the test set. A random split would be wrong here, because it would let the model learn from the future.

In [ ]:
FEATURES = [
    "DayOfWeek", "DayOfMonth", "Month", "Year", "IsWeekend",
    "Lag_1", "Lag_7", "Lag_14", "Lag_28",
    "RollingMean_7", "RollingMean_28",
]
TARGET = "DailyQuantity"

test_start = df_forecast["SalesDate"].max() - pd.Timedelta(days=TEST_DAYS)

train_data = df_forecast[df_forecast["SalesDate"] < test_start]
test_data = df_forecast[df_forecast["SalesDate"] >= test_start]

X_train, y_train = train_data[FEATURES], train_data[TARGET]
X_test, y_test = test_data[FEATURES], test_data[TARGET]

print(f"Train: {len(train_data):,} rows  ({train_data['SalesDate'].min().date()} -> {train_data['SalesDate'].max().date()})")
print(f"Test : {len(test_data):,} rows  ({test_data['SalesDate'].min().date()} -> {test_data['SalesDate'].max().date()})")
print(f"Mean demand in the test period: {y_test.mean():.1f} units/day   |   median: {y_test.median():.1f}")

The mean is far above the median: most days have modest sales and a few days have huge wholesale orders. This **spiky demand** is what makes the problem hard.

## 6. Evaluation metrics

* **MAE** – average absolute error in units.
* **RMSE** – like MAE but punishes large misses more.
* **WAPE** – total absolute error divided by total actual demand (volume-weighted percentage error).

MAPE is deliberately **not** used: many days have zero demand, which makes MAPE undefined or explosive.

In [ ]:
def forecast_metrics(y_true, y_pred) -> dict:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "WAPE_%": np.abs(y_true - y_pred).sum() / np.abs(y_true).sum() * 100,
    }

## 7. Models

### 7.1 Baseline – "same as last week"

Predict that a product sells today what it sold exactly 7 days ago. Any useful model must beat this.

In [ ]:
baseline_pred = test_data["Lag_7"]
baseline_scores = forecast_metrics(y_test, baseline_pred)
baseline_scores

### 7.2 Random Forest

In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=200, max_depth=15, random_state=RANDOM_STATE, n_jobs=-1
)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_scores = forecast_metrics(y_test, rf_pred)
rf_scores

### 7.3 XGBoost (with product identity)

XGBoost also receives a one-hot column per product (`Product_<StockCode>`) so a single model can learn each product's own level. Rows are sorted chronologically before fitting.

In [ ]:
product_dummies = pd.get_dummies(df_forecast["StockCode"], prefix="Product", dtype=int)
xgb_features = FEATURES + list(product_dummies.columns)

df_xgb = (
    pd.concat([df_forecast, product_dummies], axis=1)
    .sort_values(["SalesDate", "StockCode"])
    .reset_index(drop=True)
)

xgb_train = df_xgb[df_xgb["SalesDate"] < test_start]
xgb_test = df_xgb[df_xgb["SalesDate"] >= test_start]

xgb_model = xgb.XGBRegressor(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    objective="reg:squarederror", random_state=RANDOM_STATE, n_jobs=-1,
)
xgb_model.fit(xgb_train[xgb_features], xgb_train[TARGET])

xgb_pred = xgb_model.predict(xgb_test[xgb_features])
xgb_scores = forecast_metrics(xgb_test[TARGET], xgb_pred)
xgb_scores

## 8. Results

In [ ]:
results = pd.DataFrame(
    [baseline_scores, rf_scores, xgb_scores],
    index=["Weekly naive baseline", "Random Forest", "XGBoost"],
)
results.round(2)

In [ ]:
improvement = (1 - results.loc["XGBoost"] / results.loc["Weekly naive baseline"]) * 100
print("XGBoost vs baseline (error reduction):")
print(improvement.round(1).to_string())

**Reading the results**

* In the original run XGBoost is the best model: it beats the weekly baseline on all three metrics, most strongly on RMSE (it copes better with the large spikes).
* A WAPE close to 90 % shows the honest picture: **daily, per-product demand is hard to forecast** here. The improvement is real but modest, and the forecasts should be used as a planning aid, not as exact numbers.
* The biggest errors come from a few bulk orders (below), which no calendar or lag feature can anticipate.

## 9. Error analysis

In [ ]:
predictions = xgb_test[["SalesDate", "StockCode", TARGET]].copy()
predictions["PredictedQuantity"] = xgb_pred
predictions["AbsoluteError"] = (predictions[TARGET] - predictions["PredictedQuantity"]).abs()

print("Ten largest forecast errors:")
predictions.sort_values("AbsoluteError", ascending=False).head(10)

### Actual vs predicted: best-selling product

In [ ]:
best_seller = top_products[0]
one_product = predictions[predictions["StockCode"] == best_seller]

plt.figure(figsize=(12, 4.5))
plt.plot(one_product["SalesDate"], one_product[TARGET], label="Actual", marker="o", markersize=3)
plt.plot(one_product["SalesDate"], one_product["PredictedQuantity"], label="Predicted", marker="o", markersize=3)
plt.title(f"XGBoost demand forecast - product {best_seller}")
plt.xlabel("Date")
plt.ylabel("Units per day")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Actual vs predicted: total demand of all modelled products

In [ ]:
daily_total = predictions.groupby("SalesDate")[[TARGET, "PredictedQuantity"]].sum()

plt.figure(figsize=(12, 4.5))
plt.plot(daily_total.index, daily_total[TARGET], label="Actual", marker="o", markersize=3)
plt.plot(daily_total.index, daily_total["PredictedQuantity"], label="Predicted", marker="o", markersize=3)
plt.title("XGBoost forecast - total daily demand of the modelled products")
plt.xlabel("Date")
plt.ylabel("Units per day")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### What does XGBoost rely on?

In [ ]:
top_features = (
    pd.Series(xgb_model.feature_importances_, index=xgb_features)
    .nlargest(15)
    .sort_values()
)

plt.figure(figsize=(9, 6))
plt.barh(top_features.index, top_features.values)
plt.title("Top 15 features - XGBoost demand model")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## 10. Conclusions and next steps

* A tree-based model with lag and rolling features beats the weekly baseline, but daily demand for wholesale-driven products stays noisy.
* **Ideas to improve:** forecast **weekly** instead of daily (less noise), train on `log1p(quantity)` or use a Tweedie / Poisson objective for count-like data, add price and holiday/seasonality features, and add a multi-step (30-day-ahead) evaluation.

## 11. Export for the web app

The hold-out forecasts (actual, XGBoost and baseline) and the metrics are saved so the app can show them without re-training anything.

In [ ]:
ARTIFACTS = root / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)

descriptions = read_sql(
    "SELECT StockCode, MAX(Description) AS Description FROM dbo.vw_NormalSales GROUP BY StockCode"
)

export = predictions.copy()
export["BaselineQuantity"] = xgb_test["Lag_7"]          # same index as predictions
export = (
    export.merge(descriptions, on="StockCode", how="left")
    .rename(columns={TARGET: "ActualQuantity"})
    [["SalesDate", "StockCode", "Description", "ActualQuantity", "PredictedQuantity", "BaselineQuantity"]]
)
export.to_csv(ARTIFACTS / "demand_forecasts.csv", index=False)

results.round(2).reset_index(names="Model").to_csv(ARTIFACTS / "forecast_metrics.csv", index=False)

print("Saved:", sorted(f.name for f in ARTIFACTS.glob("*")))